# GraphRAG — Person B Smoke Test
Run each cell top to bottom. Each cell has a ✅ / ❌ check at the end so you know immediately if something is broken.

> **Before starting:** Make sure the `GraphRAG/` folder is uploaded to `MyDrive/GraphRAG/`

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, os

PROJECT_ROOT = '/content/drive/MyDrive/GraphRAG'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print('Working directory:', os.getcwd())
print('Files found:', os.listdir('.'))
print('\n✅ Drive mounted and project root set')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────
# This takes 2-3 minutes on first run. Safe to skip on reruns.
!pip install pyyaml python-dotenv spacy tiktoken \
             google-generativeai groq \
             networkx chromadb \
             graspologic leidenalg \
             pymupdf sentence-transformers \
             streamlit pyvis -q

!python -m spacy download en_core_web_sm -q

print('\n✅ Dependencies installed')

In [ ]:
# ── Cell 3: Set API keys ──────────────────────────────────────
# Paste your keys here. This writes a .env file to the project root.
# Do NOT share this notebook with keys still in it.

GEMINI_API_KEY = 'paste_your_gemini_key_here'
GROQ_API_KEY   = 'paste_your_groq_key_here'   # optional

env_content = f'GEMINI_API_KEY={GEMINI_API_KEY}\nGROQ_API_KEY={GROQ_API_KEY}\n'
with open(f'{PROJECT_ROOT}/.env', 'w') as f:
    f.write(env_content)

print('.env written to project root')
print('Gemini key set:', bool(GEMINI_API_KEY and GEMINI_API_KEY != "paste_your_gemini_key_here"))
print('\n✅ API keys saved')

In [ ]:
# ── Cell 4: Test config loader ────────────────────────────────
from src.config import cfg, GEMINI_API_KEY, CACHE_DIR, CHROMA_DIR, GRAPH_PATH

print('Config sections:', list(cfg.keys()))
print('Chunk size:', cfg['chunking']['chunk_size'])
print('LLM provider:', cfg['llm']['generation_provider'])
print('Cache dir:', CACHE_DIR)
print('Graph path:', GRAPH_PATH)
print('Gemini key loaded:', bool(GEMINI_API_KEY))

assert 'chunking' in cfg, 'chunking section missing from config'
assert cfg['chunking']['chunk_size'] == 600, 'unexpected chunk size'
print('\n✅ Config loads correctly')

In [ ]:
# ── Cell 5: Test chunker ──────────────────────────────────────
from src.chunker import chunk_document, count_tokens, split_into_sentences

sample_text = """
The transformer architecture was introduced in the paper Attention is All You Need.
It relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions entirely.
The encoder maps an input sequence to a sequence of continuous representations.
Given these representations, the decoder then generates an output sequence one element at a time.
Self-attention allows each position in the sequence to attend to all positions in the previous layer.
This property is key to capturing long-range dependencies efficiently and in parallel.
Multi-head attention allows the model to jointly attend to information from different representation subspaces.
Position-wise feed-forward networks are applied to each position separately and identically.
The model uses residual connections around each sub-layer followed by layer normalization.
Positional encodings are added to the input embeddings to inject sequence order information.
""" * 15  # repeat to generate enough tokens for multiple chunks

# Test sentence splitting
sentences = split_into_sentences(sample_text)
print(f'Sentences found: {len(sentences)}')
print(f'First sentence: {sentences[0][:80]}')

# Test chunking
chunks = chunk_document(sample_text, doc_id='test_001')
print(f'\nChunks produced: {len(chunks)}')

for c in chunks[:3]:
    print(f"  [{c['chunk_index']}] tokens={c['token_count']}  sentences={c['sentences']}  id={c['chunk_id']}")

assert len(chunks) > 1, 'Expected multiple chunks from long text'
assert all(c['token_count'] <= 650 for c in chunks), 'Chunk exceeded token limit'
assert all('chunk_id' in c for c in chunks), 'Missing chunk_id field'
print('\n✅ Chunker works correctly')

In [ ]:
# ── Cell 6: Test LLM client ───────────────────────────────────
from src.llm_client import LLMClient

llm = LLMClient(purpose='generation')
print('Provider:', llm.provider)
print('Model:', llm.model)

response = llm.generate('In one sentence, what is a knowledge graph?')
print('\nLLM response:', response)

assert len(response) > 10, 'Response too short — API may have failed'
print('\n✅ LLM client works')

In [ ]:
# ── Cell 7: Test embeddings ───────────────────────────────────
embedding = llm.embed('attention mechanism in transformers')

print('Embedding type:', type(embedding))
print('Embedding dimensions:', len(embedding))
print('First 5 values:', [round(v, 4) for v in embedding[:5]])

assert isinstance(embedding, list), 'Embedding should be a list'
assert len(embedding) > 100, 'Embedding vector too short'
print('\n✅ Embeddings work')

In [ ]:
# ── Cell 8: Test vector store ─────────────────────────────────
from src.vector_store import VectorStore

vs = VectorStore()
print('Initial stats:', vs.stats())

# Index the test chunks from Cell 5
vs.build_index(chunks)
print('After indexing:', vs.stats())

# Query
results = vs.query('attention mechanism transformer')
print(f'\nQuery returned {len(results)} results:')
for r in results[:2]:
    print(f"  score={r['score']}  doc={r['doc_id']}")
    print(f"  text: {r['text'][:80]}...")

assert len(results) > 0, 'Vector store returned no results'
assert all('score' in r for r in results), 'Missing score field'
print('\n✅ Vector store works')

In [ ]:
# ── Cell 9: Test router ───────────────────────────────────────
from src.router import QueryRouter

router = QueryRouter(llm=llm)

test_queries = [
    ('What datasets were used in the BERT paper?',       'LOCAL'),
    ('What are the main themes across all papers?',      'GLOBAL'),
    ('Explain attention and its role in the field',      'HYBRID'),
]

print('Router test results:')
print('-' * 60)
for query, expected in test_queries:
    decision = router.route(query)
    match = '✅' if decision.route_type.value == expected else '⚠️ '
    print(f"{match} [{decision.route_type.value:6s}] conf={decision.confidence:.2f}  '{query[:45]}'")

print('\n(⚠️  means router disagreed with expected — not necessarily wrong, LLMs can vary)')
print('\n✅ Router runs without errors')

In [ ]:
# ── Cell 10: Test community detection on toy graph ────────────
# This does NOT need Person A's graph — uses a synthetic test graph.
import networkx as nx
from src.community_detection import CommunityDetector
from src.config import GRAPH_PATH

# Build a small synthetic knowledge graph for testing
G = nx.MultiDiGraph()

# Cluster 1: NLP / Transformers
nlp_nodes = ['transformer', 'attention', 'BERT', 'GPT', 'self-attention', 'encoder', 'decoder']
# Cluster 2: Computer Vision
cv_nodes  = ['CNN', 'ResNet', 'image_classification', 'convolution', 'pooling']
# Cluster 3: Training
tr_nodes  = ['backpropagation', 'gradient_descent', 'Adam_optimizer', 'learning_rate']

for n in nlp_nodes + cv_nodes + tr_nodes:
    G.add_node(n, description=f'Entity: {n}', entity_type='concept', frequency=1)

# Add edges within clusters (dense)
for i in range(len(nlp_nodes)-1):
    G.add_edge(nlp_nodes[i], nlp_nodes[i+1],
               relation_type='related_to', description='', weight=1.0, computed_weight=1.0)
G.add_edge('BERT', 'transformer', relation_type='based_on', description='', weight=2.0, computed_weight=2.0)
G.add_edge('GPT',  'transformer', relation_type='based_on', description='', weight=2.0, computed_weight=2.0)

for i in range(len(cv_nodes)-1):
    G.add_edge(cv_nodes[i], cv_nodes[i+1],
               relation_type='related_to', description='', weight=1.0, computed_weight=1.0)

for i in range(len(tr_nodes)-1):
    G.add_edge(tr_nodes[i], tr_nodes[i+1],
               relation_type='related_to', description='', weight=1.0, computed_weight=1.0)

# Cross-cluster (sparse)
G.add_edge('transformer', 'Adam_optimizer', relation_type='trained_with',
           description='', weight=0.5, computed_weight=0.5)
G.add_edge('ResNet', 'backpropagation', relation_type='trained_with',
           description='', weight=0.5, computed_weight=0.5)

# Save as GML so CommunityDetector can load it
import os
os.makedirs(GRAPH_PATH.parent, exist_ok=True)
nx.write_gml(G, str(GRAPH_PATH))
print(f'Toy graph saved: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Run community detection
cd = CommunityDetector()
cd.load_graph()
communities = cd.detect_all_levels()

print('\nCommunity stats:')
for res, stats in cd.stats().items():
    print(f'  Resolution {res}: {stats}')

# Check resolution 1.0
members_1 = cd.get_community_members(1.0)
print(f'\nCommunities at resolution=1.0:')
for cid, members in members_1.items():
    print(f'  Community {cid}: {members}')

assert len(members_1) >= 2, 'Expected at least 2 communities'
print('\n✅ Community detection works')

In [ ]:
# ── Cell 11: Test community summarizer (1 community) ─────────
from src.community_summarizer import CommunitySummarizer

cs = CommunitySummarizer(detector=cd, llm=llm)

# Summarize just one community to test (not all — saves API quota)
first_comm_id = list(members_1.keys())[0]
first_members = members_1[first_comm_id]

print(f'Summarizing community {first_comm_id} with {len(first_members)} members...')
summary = cs._summarize_community(first_comm_id, first_members, resolution=1.0)

print('\nSummary preview:')
print(summary['summary'][:300], '...')

assert 'summary' in summary, 'Missing summary field'
assert len(summary['summary']) > 50, 'Summary too short'
print('\n✅ Community summarizer works')

In [ ]:
# ── Cell 12: Full end-to-end query test ───────────────────────
# Runs the complete pipeline: route → retrieve → answer
from src.query_engine import QueryEngine
from src.graph_query import GraphQueryEngine

# First: generate summaries for all communities (needed by graph engine)
print('Generating all community summaries...')
all_summaries = cs.summarize_all(resolution=1.0)
print(f'Generated {len(all_summaries)} summaries')

# Build graph engine
graph_engine = GraphQueryEngine(summarizer=cs, llm=llm)

# Build unified engine
engine = QueryEngine(
    vector_store=vs,
    graph_engine=graph_engine,
    router=router,
    llm=llm,
)

# Test all three routes
test_cases = [
    ('What is self-attention?',                    'LOCAL'),
    ('What are the main themes in this corpus?',   'GLOBAL'),
    ('Explain transformers and their training',    'HYBRID'),
]

print('\nEnd-to-end query test:')
print('=' * 60)
for query, forced_route in test_cases:
    print(f'\nQuery: {query}')
    print(f'Forced route: {forced_route}')
    result = engine.query(query, force_route=forced_route)
    print(f'Route taken: {result["route"]}')
    print(f'Answer preview: {result["answer"][:150]}...')
    assert result['answer'], 'Empty answer'

print('\n✅ Full pipeline works end-to-end')

In [ ]:
# ── Cell 13: Summary ──────────────────────────────────────────
print('=' * 50)
print('SMOKE TEST COMPLETE')
print('=' * 50)
print('''
All components verified:
  Cell 4  — Config loader
  Cell 5  — Sentence-aware chunker
  Cell 6  — LLM client (Gemini)
  Cell 7  — Embeddings
  Cell 8  — Vector store (ChromaDB)
  Cell 9  — Hybrid query router
  Cell 10 — Community detection (Leiden)
  Cell 11 — Community summarizer
  Cell 12 — Full end-to-end pipeline

What to do next:
  1. Wait for Person A to share knowledge_graph.gml
  2. Replace the toy graph with the real one
  3. Re-run cells 10-12 on the real graph
  4. Run: streamlit run app.py
''')